# 06 — Conversational Interface (Module D)
**NewsBot Intelligence System 2.0 | ITAI 2373**

- D.1 Intent Classification
- D.2 Natural Language Query Processing
- D.3 Context Management
- D.4 Response Generation

In [ ]:
!pip install requests -q

In [ ]:
import re, json, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a1d2e',
    'axes.edgecolor':'#2d3561','axes.labelcolor':'#e8d5a3',
    'xtick.color':'#a0a8c0','ytick.color':'#a0a8c0',
    'text.color':'#e8d5a3','grid.color':'#2d3561','grid.alpha':0.5,'figure.dpi':120
})
CATEGORIES = ['tech','business','politics','sport','entertainment']
PALETTE = {'tech':'#4fc3f7','business':'#81c784','politics':'#e57373',
           'sport':'#ffb74d','entertainment':'#ce93d8'}
print('Module D setup complete.')


In [ ]:
try:
    _ = df_final
    print(f'df_final: {len(df_final)} articles')
except NameError:
    raise RuntimeError('Run notebooks 01 and 02 first.')


## D.1 — Intent Classification

In [ ]:
INTENT_PATTERNS = {
    'filter_by_sentiment': [
        r'positive\s+news', r'negative\s+news', r'neutral\s+news',
        r'optimistic', r'pessimistic', r'upbeat', r'bad news'
    ],
    'filter_by_category': [rf'\b{cat}\b' for cat in CATEGORIES],
    'summarize':           [r'summar', r'brief', r'tldr', r'overview', r'gist'],
    'top_entities':        [r'who.*mentioned', r'most.*mentioned', r'top.*entit'],
    'topic_query':         [r'topic', r'theme', r'about\s+what', r'main\s+subject'],
    'search':              [r'find.*articles?', r'search\s+for', r'show\s+me', r'articles?.*about'],
    'stats':               [r'how\s+many', r'count', r'statistics', r'breakdown', r'distribution'],
    'comparison':          [r'compar', r'vs\.?', r'versus', r'difference\s+between'],
}

def classify_intent(query):
    q = query.lower()
    for intent, patterns in INTENT_PATTERNS.items():
        for p in patterns:
            if re.search(p, q):
                return intent, 0.95
    return 'general', 0.50

def extract_filters(query):
    q = query.lower()
    filters = {}
    for cat in CATEGORIES:
        if cat in q:
            filters['category'] = cat
            break
    if any(w in q for w in ['positive','optimistic','upbeat']):
        filters['sentiment_label'] = 'Positive'
    elif any(w in q for w in ['negative','pessimistic']):
        filters['sentiment_label'] = 'Negative'
    elif 'neutral' in q:
        filters['sentiment_label'] = 'Neutral'
    kw = re.search(r'(?:about|on|covering|regarding)\s+([a-zA-Z\s]+?)(?:\s+from|\s+in|\s+this|$)', q)
    if kw:
        filters['keyword'] = kw.group(1).strip()
    return filters

# Demo intent classification
print('D.1 — Intent Classification Demo')
print('='*55)
test_queries = [
    'Show me positive tech news',
    'How many business articles are there?',
    'Find articles about climate change',
    'Compare sports vs entertainment sentiment',
    'Summarize the top politics story',
    'Who is mentioned most in tech articles?',
    'What topics appear in business news?',
]
for q in test_queries:
    intent, conf = classify_intent(q)
    filters = extract_filters(q)
    print(f'  Query  : {q}')
    print(f'  Intent : {intent} ({conf*100:.0f}% confidence) | Filters: {filters}')
    print()


## D.2 — Natural Language Query Execution

In [ ]:
def execute_query(query, df):
    intent, conf  = classify_intent(query)
    filters       = extract_filters(query)
    subset        = df.copy()

    if 'category' in filters:
        subset = subset[subset.category==filters['category']]
    if 'sentiment_label' in filters and 'sentiment_label' in subset.columns:
        subset = subset[subset.sentiment_label==filters['sentiment_label']]
    if 'keyword' in filters:
        col = 'cleaned_text' if 'cleaned_text' in subset.columns else 'text'
        subset = subset[subset[col].str.lower().str.contains(filters['keyword'], na=False)]

    n = len(subset)
    result = {'intent':intent,'filters':filters,'n_results':n,'query':query}

    if n == 0:
        result['response'] = f'No articles matched: "{query}"'
    elif intent == 'stats':
        dist = subset.category.value_counts().to_dict()
        result['response'] = f'Found {n} articles. Category breakdown: {dist}'
    elif intent == 'filter_by_sentiment':
        label = filters.get('sentiment_label','all sentiments')
        avg   = subset.sentiment_compound.mean() if 'sentiment_compound' in subset.columns else 'N/A'
        result['response'] = f'Found {n} {label.lower()} articles. Avg sentiment: {avg:.3f}' if isinstance(avg,float) else f'Found {n} articles.'
    elif intent == 'top_entities':
        from collections import Counter
        if 'entities' in subset.columns:
            freq = Counter()
            for ents in subset.entities:
                for lst in ents.values(): freq.update(lst)
            top5 = freq.most_common(5)
            result['response'] = f'Top entities: {[e for e,_ in top5]}'
        else:
            result['response'] = f'Found {n} articles.'
    else:
        result['response'] = f'Found {n} articles matching your query.'

    return result

# Demo queries
print('D.2 — Query Execution Demo')
print('='*55)
for q in [
    'Show me positive tech news',
    'How many business articles are there?',
    'Find articles about inflation',
    'Who is most mentioned in politics?',
]:
    r = execute_query(q, df_final)
    print(f'Q: {q}')
    print(f'A: {r["response"]}  [{r["n_results"]} articles found]')
    print()


## D.3 — Context Management (Multi-turn Corpus Chat)

In [ ]:
OLLAMA_HOST  = 'http://127.0.0.1:11434'
OLLAMA_MODEL = 'llama3.2:1b'

def call_llm(prompt, system=None):
    messages = []
    if system: messages.append({'role':'system','content':system})
    messages.append({'role':'user','content':prompt})
    try:
        r = requests.post(f'{OLLAMA_HOST}/api/chat',
            json={'model':OLLAMA_MODEL,'messages':messages,
                  'stream':False,'options':{'temperature':0.3,'num_predict':400}},
            timeout=60)
        r.raise_for_status()
        return r.json()['message']['content'].strip()
    except Exception as e:
        return f'[LLM unavailable: {e}]'

class NewsBotConversation:
    """Multi-turn conversational interface over the full BBC corpus."""
    SYSTEM = ('You are NewsBot 2.0, an intelligent news analysis assistant. '
              'You have access to 2,225 BBC news articles across 5 categories. '
              'Answer questions concisely and accurately. '
              'When referencing data, be specific about numbers and categories.')

    def __init__(self, df):
        self.df       = df
        self._history = []
        self._stats   = {
            'total':      len(df),
            'categories': df.category.value_counts().to_dict(),
            'sentiment':  df.sentiment_label.value_counts().to_dict() if 'sentiment_label' in df.columns else {}
        }

    def chat(self, user_message):
        # Try structured query first
        qr = execute_query(user_message, self.df)
        structured_ctx = f'[Query result: {qr["response"]}]' if qr['n_results'] > 0 else ''

        history_str = ''
        if self._history:
            history_str = '\n'.join(
                f'User: {t["user"]}\nNewsBot: {t["bot"]}' for t in self._history[-3:]
            )
            history_str = f'\nPrevious conversation:\n{history_str}\n'

        corpus_ctx = (f'Dataset stats: {self._stats["total"]} articles, '
                      f'categories: {self._stats["categories"]}')

        prompt = (f'{self.SYSTEM}\n\n{corpus_ctx}\n{history_str}'
                  f'{structured_ctx}\nUser: {user_message}\nNewsBot:')
        response = call_llm(prompt)
        self._history.append({'user': user_message, 'bot': response})
        return response

    def show_history(self):
        print('='*55)
        for i,t in enumerate(self._history,1):
            print(f'[{i}] You: {t["user"]}')
            print(f'     NewsBot: {t["bot"]}\n')

# Demo multi-turn conversation
print('D.3 — Multi-turn Corpus Conversation Demo')
print('='*55)
bot = NewsBotConversation(df_final)

turns = [
    'How many articles are in the dataset?',
    'Which category has the most positive sentiment?',
    'Tell me more about the tech articles.',
]
for msg in turns:
    print(f'You: {msg}')
    reply = bot.chat(msg)
    print(f'NewsBot: {reply}\n')


## D.4 — Query Analytics & Visualization

In [ ]:
# Run a set of benchmark queries and visualize intent distribution
benchmark_queries = [
    'Show me positive tech news',
    'How many business articles are there?',
    'Find articles about interest rates',
    'Who is mentioned most in politics?',
    'Compare sports vs entertainment',
    'Summarize the latest news',
    'What topics appear in business news?',
    'Show me negative sentiment articles',
    'Find articles about the economy',
    'How many sport articles mention championship?',
]

intent_counts = {}
query_results = []
for q in benchmark_queries:
    intent, conf = classify_intent(q)
    intent_counts[intent] = intent_counts.get(intent, 0) + 1
    r = execute_query(q, df_final)
    query_results.append({'query':q,'intent':intent,'n_results':r['n_results'],'confidence':conf})

results_df = pd.DataFrame(query_results)
print('Benchmark Query Results:')
print(results_df[['query','intent','n_results']].to_string(index=False))


In [ ]:
# Intent distribution chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

# Intent pie
intents = list(intent_counts.keys())
counts  = list(intent_counts.values())
colors  = plt.cm.Set2(np.linspace(0,1,len(intents)))
ax1.pie(counts, labels=[i.replace('_',' ').title() for i in intents],
        colors=colors, autopct='%1.0f%%', startangle=90)
ax1.set_title('Intent Distribution', fontsize=12, fontweight='bold')

# Results per query bar
ax2.barh(range(len(results_df)), results_df.n_results,
         color=[PALETTE.get(c,'#a0a8c0') for c in ['tech','business','politics','sport','entertainment']*2])
ax2.set_yticks(range(len(results_df)))
ax2.set_yticklabels([q[:35]+'...' if len(q)>35 else q for q in results_df.query], fontsize=8)
ax2.set_title('Articles Returned per Query', fontsize=12, fontweight='bold')
ax2.set_xlabel('Article Count')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('query_analytics.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


In [ ]:
print('='*55)
print('  MODULE D SUMMARY')
print('='*55)
print(f'  Intent categories  : {len(INTENT_PATTERNS)}')
print(f'  Queries tested     : {len(benchmark_queries)}')
print(f'  Intent accuracy    : Rule-based (95% confidence on match)')
print(f'  Context window     : Last 3 conversation turns')
print(f'  LLM backend        : ollama/{OLLAMA_MODEL}')
print('='*55)
